# Notebook 3: Inference — Load Fine-tuned Model and Generate Agent Responses

Load the base GPT-OSS-120B model plus the LoRA adapter trained in `02_sft_training.ipynb`,
and expose a single function `generate_agent_response(system_prompt, context, conversation_history)`
that returns the next agent utterance as a plain string.

**Inputs to the generation function:**
- `system_prompt` (str): the system instruction for the agent.
- `context` (str): retrieval/knowledge context — previously an empty placeholder, now fully usable. Pass `""` to omit.
- `conversation_history` (list of `(speaker, text)` tuples): prior turns, where `speaker ∈ {"agent", "customer"}`.

**Prerequisites:** `checkpoints/final_adapter/` saved by notebook 2.

## Configuration

In [ ]:
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ============================================================
# Update these two paths before running
# ============================================================
MODEL_DIR = "/path/to/gpt-oss-120b"          # <-- base model directory (same as notebook 2)
ADAPTER_PATH = "checkpoints/final_adapter"    # <-- LoRA adapter dir saved by notebook 2

# Generation defaults
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.7
TOP_P = 0.9

## Load Tokenizer

Load from `ADAPTER_PATH` — notebook 2 saved the tokenizer there with the 8 registered special tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, local_files_only=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

AGENT_TOKEN_ID    = tokenizer.convert_tokens_to_ids("<|agent|>")
CUSTOMER_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|customer|>")
CONV_END_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|/conversation|>")
STOP_TOKEN_IDS    = [CUSTOMER_TOKEN_ID, CONV_END_TOKEN_ID, tokenizer.eos_token_id]

print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"<|agent|>={AGENT_TOKEN_ID}, <|customer|>={CUSTOMER_TOKEN_ID}, <|/conversation|>={CONV_END_TOKEN_ID}")

## Load Base Model in 4-bit + Attach LoRA Adapter

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
base_model.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("Model + adapter loaded and set to eval mode.")

## Generation Function

Builds the same prompt template used during training:

```
<|system|>{system_prompt}<|/system|>
<|context|>{context}<|/context|>
<|conversation|>
<|agent|>...
<|customer|>...
<|/conversation|>
<|agent|>
```

Generation halts on `<|customer|>`, `<|/conversation|>`, or EOS.

In [ ]:
def _format_conversation(history):
    """Turn a list of (speaker, text) tuples into the training-format conversation block."""
    lines = []
    for speaker, text in history:
        if speaker not in ("agent", "customer"):
            raise ValueError(f"speaker must be 'agent' or 'customer', got {speaker!r}")
        marker = "<|agent|>" if speaker == "agent" else "<|customer|>"
        lines.append(f"{marker}{text}")
    return "\n".join(lines)


def build_prompt(system_prompt: str, context: str, conversation_history: list) -> str:
    """Assemble the prompt string that ends with a trailing <|agent|> tag."""
    conv_str = _format_conversation(conversation_history)
    # Trailing newline between conv and </conversation> only when history exists,
    # otherwise keep the block compact.
    conv_block = f"{conv_str}\n" if conv_str else ""
    return (
        f"<|system|>{system_prompt}<|/system|>\n"
        f"<|context|>{context}<|/context|>\n"
        f"<|conversation|>\n"
        f"{conv_block}"
        f"<|/conversation|>\n"
        f"<|agent|>"
    )


def generate_agent_response(
    system_prompt: str,
    context: str = "",
    conversation_history: list | None = None,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    do_sample: bool = True,
) -> str:
    """Generate the next agent utterance.

    conversation_history: list of (speaker, text) pairs, speaker in {'agent','customer'}.
    Pass [] (or None) to have the model produce the opening agent line.
    """
    history = conversation_history or []
    prompt = build_prompt(system_prompt, context, history)

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            eos_token_id=STOP_TOKEN_IDS,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids = output_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=False)

    # Defensive trim in case the model emits the next turn marker
    for stop_tok in ["<|customer|>", "<|/conversation|>", "<|agent|>", tokenizer.eos_token]:
        if stop_tok and stop_tok in response:
            response = response.split(stop_tok, 1)[0]

    return response.strip()

## Example Usage

Swap in any system prompt, context string, and conversation history. Re-run the cell to sample a new response.

In [ ]:
system_prompt = (
    "You are an outbound sales agent for American Express.\n"
    "Your goal is to identify customer needs and guide the conversation\n"
    "toward the AP automation product. Use the context block when available."
)

context = (
    "Current campaign: AP automation for mid-market B2B.\n"
    "Key benefits: 60% faster invoice processing, integration with major ERPs,\n"
    "2% cashback on eligible spend."
)

conversation_history = [
    ("agent", "Hi, this is Alex from American Express. Do you have a minute to talk about streamlining your AP workflow?"),
    ("customer", "We already use a tool for that. What makes yours different?"),
]

response = generate_agent_response(
    system_prompt=system_prompt,
    context=context,
    conversation_history=conversation_history,
)
print("Agent:", response)

### Empty history: generate the opening line

Pass an empty `conversation_history` (and any context or blank) to have the model produce the agent's first utterance.

In [ ]:
opening = generate_agent_response(
    system_prompt=system_prompt,
    context="",
    conversation_history=[],
)
print("Agent (opening):", opening)

## Interactive Chat: Simulate an Outbound Sales Call

The model generates the opening agent line, then you play the customer. Type `quit` to end the call.
Edit `system_prompt` / `context` in the cell above before running.

In [ ]:
conversation_turns = []  # list of (speaker, text)

print("=" * 60)
print("OUTBOUND SALES CALL SIMULATION")
print("You are the customer. The agent will speak first.")
print("Type 'quit' to end the conversation.")
print("=" * 60)

while True:
    agent_text = generate_agent_response(
        system_prompt=system_prompt,
        context=context,
        conversation_history=conversation_turns,
    )
    print(f"\nAgent: {agent_text}")
    conversation_turns.append(("agent", agent_text))

    user_input = input("\nCustomer: ").strip()
    if user_input.lower() == "quit":
        print("\n[Call ended]")
        break

    conversation_turns.append(("customer", user_input))

## Base Model Only (No Adapter) — Baseline Comparison

Same prompt template, but the LoRA adapter is temporarily disabled via
`model.disable_adapter()`. Useful for comparing fine-tuned vs. base behavior
on identical inputs without reloading weights.

In [ ]:
def generate_base_response(
    system_prompt: str,
    context: str = "",
    conversation_history: list | None = None,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    do_sample: bool = True,
) -> str:
    """Generate the next agent utterance using the BASE model only (adapter disabled).

    Same signature and prompt template as generate_agent_response, but wraps the
    generate call in model.disable_adapter() so the LoRA weights are not applied.
    """
    history = conversation_history or []
    prompt = build_prompt(system_prompt, context, history)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad(), model.disable_adapter():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            eos_token_id=STOP_TOKEN_IDS,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids = output_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=False)
    for stop_tok in ["<|customer|>", "<|/conversation|>", "<|agent|>", tokenizer.eos_token]:
        if stop_tok and stop_tok in response:
            response = response.split(stop_tok, 1)[0]
    return response.strip()


# Example: base-model baseline on the same inputs used in the earlier example cell.
# Run the A/B side by side to see how the adapter changes behavior.
base_response = generate_base_response(
    system_prompt=system_prompt,
    context=context,
    conversation_history=conversation_history,
)
ft_response = generate_agent_response(
    system_prompt=system_prompt,
    context=context,
    conversation_history=conversation_history,
)
print("Base model:   ", base_response)
print()
print("Fine-tuned:   ", ft_response)